# Importador de históricos ChileCompra

**Versión 1.0 — carga de `Procesos` al clon exclusivamente.** No llama a las API, no ejecuta GitHub Actions y no modifica la base original. El histórico se lee sin alterarlo.

Flujo: autenticar → subir ZIP/CSV → normalizar → simular → autorizar carga → respaldar → escribir por bloques → verificar.

**Por defecto está en simulación.**

In [ ]:
!pip -q install pandas gspread google-api-python-client google-auth

In [ ]:
from google.colab import auth, files
from google.auth import default
import gspread, pandas as pd, zipfile, io, os, re, json, datetime, unicodedata, time
from googleapiclient.discovery import build
auth.authenticate_user()
creds,_=default(scopes=['https://www.googleapis.com/auth/spreadsheets','https://www.googleapis.com/auth/drive'])
gc=gspread.authorize(creds)
drive=build('drive','v3',credentials=creds)

## Configuración
El destino está fijado al identificador del clon aprobado. Primero deja `EJECUTAR_ESCRITURA=False`.

In [ ]:
CLON_ID='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
HOJA='Procesos'
FUENTE='licitaciones' # licitaciones | compra_agil
EJECUTAR_ESCRITURA=False
CONFIRMACION=''
LOTE=300
COLUMNAS=['codigo','nombre','estado','convocatoria','fecha_publicacion','fecha_cierre','monto_disponible_clp','organismo_comprador','region','total_ofertas_recibidas','motivo_seleccion','url_detalle','fecha_sincronizacion','categoria','subcategoria','modalidad']
assert FUENTE in ('licitaciones','compra_agil')
assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
ws=gc.open_by_key(CLON_ID).worksheet(HOJA)
assert ws.row_values(1)==COLUMNAS,'Encabezados incompatibles'
print('Destino validado: clon, Procesos')

## Subir archivo
Selecciona **un ZIP o CSV** por ejecución; no mezcles fuentes.

In [ ]:
archivos=files.upload()
assert len(archivos)==1,'Sube un único ZIP/CSV'
nombre_archivo=next(iter(archivos))
assert nombre_archivo.lower().endswith(('.zip','.csv'))
ruta='/content/'+nombre_archivo

In [ ]:
def normal(s):
 s=unicodedata.normalize('NFKD',str(s))
 return re.sub('[^a-z0-9]','', ''.join(c for c in s if not unicodedata.combining(c)).lower())
ALIAS={
'codigo':['codigoexterno','codigolicitacion','codigocotizacion','codigoagil','codigo'],
'nombre':['nombre','nombrelicitacion','nombrecotizacion','titulo'],
'estado':['estado','estadolicitacion','estadoproceso'],
'convocatoria':['convocatoria'],
'fecha_publicacion':['fechapublicacion','fechacreacion','fechaingreso'],
'fecha_cierre':['fechacierre','fechacierreofertas','fechafinalizacion'],
'monto_disponible_clp':['montodisponible','montodisponibleclp','montoestimado','presupuestoestimado'],
'organismo_comprador':['organismocomprador','nombreorganismo','nombreunidadcompra','comprador'],
'region':['region','regioncomprador'],
'total_ofertas_recibidas':['totalofertasrecibidas','cantidadofertas'],
'motivo_seleccion':['motivoseleccion'],
'url_detalle':['urldetalle','link'],
'categoria':['categoria'],'subcategoria':['subcategoria']}
def limpiar(x):
 s=str(x).strip()
 return '' if s.lower() in ('nan','none','null','nat','<na>') else s
def fecha(x):
 s=limpiar(x)
 if not s:return ''
 d=pd.to_datetime(s,errors='coerce',dayfirst=not bool(re.match(r'^\d{4}-',s)))
 return d.strftime('%Y-%m-%d') if pd.notna(d) else ''
def numero(x):
 s=re.sub(r'[$\s]','',limpiar(x))
 if not s:return ''
 if re.fullmatch(r'-?\d{1,3}(\.\d{3})+(,\d+)?',s):s=s.replace('.','').replace(',','.')
 elif ',' in s and '.' not in s:s=s.replace(',','.')
 try:return float(s) if '.' in s else int(s)
 except ValueError:return ''
def codigo(x):return limpiar(x).upper()
def fuentes(ruta):
 if ruta.lower().endswith('.csv'):
  with open(ruta,'rb') as f:yield os.path.basename(ruta),f.read()
 else:
  with zipfile.ZipFile(ruta) as z:
   items=[n for n in z.namelist() if n.lower().endswith('.csv') and not n.startswith('__MACOSX/')]
   assert items,'El ZIP no contiene CSV'
   for n in items:yield n,z.read(n)
def dialecto(data):
 for enc in ('utf-8-sig','utf-8','cp1252','latin-1'):
  try:s=data[:32768].decode(enc)
  except UnicodeDecodeError:continue
  first=s.splitlines()[0];sep=max(';,|\t',key=lambda c:first.count(c))
  if first.count(sep):return enc,sep
 raise ValueError('No se detectó delimitador')
def mapeo(cols):
 d={normal(c):c for c in cols}
 m={k:next((d[a] for a in aliases if a in d),None) for k,aliases in ALIAS.items()}
 assert m['codigo'] and m['nombre'],f'Faltan código/nombre: {cols}'
 return m
def transformar(fila,m):
 r={k:limpiar(fila.get(c,'')) if c else '' for k,c in m.items()}
 r['codigo']=codigo(r['codigo'])
 r['fecha_publicacion']=fecha(r['fecha_publicacion'])
 r['fecha_cierre']=fecha(r['fecha_cierre'])
 for k in ('monto_disponible_clp','total_ofertas_recibidas'):r[k]=numero(r[k])
 r['fecha_sincronizacion']=''
 r['modalidad']='Compra Ágil' if FUENTE=='compra_agil' else 'Licitación'
 return {k:r.get(k,'') for k in COLUMNAS}

## Simulación y reporte
Deduplicación por código; cada repetición del histórico se consolida en un proceso para `Procesos`.

In [ ]:
procesos={};originales=0;sin_codigo=0;diagnostico=[]
for archivo,raw in fuentes(ruta):
 enc,sep=dialecto(raw)
 reader=pd.read_csv(io.BytesIO(raw),encoding=enc,sep=sep,dtype=str,keep_default_na=False,chunksize=20000,on_bad_lines='error')
 n=0
 for chunk in reader:
  if n==0:
   m=mapeo(list(chunk.columns))
   diagnostico.append(dict(archivo=archivo,filas=None,encoding=enc,separador=sep,mapeo=m))
  for fila in chunk.to_dict('records'):
   originales+=1;n+=1
   r=transformar(fila,m);k=r['codigo']
   if not k:sin_codigo+=1;continue
   score=sum(bool(r[f]) for f in COLUMNAS if f not in ('codigo','modalidad'))
   prev=procesos.get(k)
   if prev is None or score>prev[0]:procesos[k]=(score,r)
 diagnostico[-1]['filas']=n
 print(archivo,n,'filas')
assert procesos,'No se encontraron procesos utilizables'
cargados=[codigo(k) for k in ws.col_values(1)[1:] if codigo(k)]
assert len(cargados)==len(set(cargados)),'La hoja contiene códigos duplicados'
existentes=set(cargados)
pendientes=[t[1] for k,t in procesos.items() if k not in existentes]
reporte={'archivo':nombre_archivo,'fuente':FUENTE,'filas_originales':originales,'sin_codigo':sin_codigo,
         'codigos_unicos':len(procesos),'repeticiones':originales-sin_codigo-len(procesos),
         'existentes':len(procesos)-len(pendientes),'nuevos':len(pendientes),
         'total_inicial':len(existentes),'total_esperado':len(existentes)+len(pendientes),
         'diagnostico':diagnostico}
print(json.dumps(reporte,ensure_ascii=False,indent=2))
display(pd.DataFrame(pendientes[:5],columns=COLUMNAS))
sufijo=datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
ruta_reporte=f'/content/simulacion_{sufijo}.json'
ruta_csv=f'/content/nuevos_{sufijo}.csv'
with open(ruta_reporte,'w',encoding='utf8') as f:json.dump(reporte,f,ensure_ascii=False,indent=2)
pd.DataFrame(pendientes,columns=COLUMNAS).to_csv(ruta_csv,index=False,encoding='utf-8-sig')

## Escritura manual opcional
Solo se activa con `EJECUTAR_ESCRITURA=True` y `CONFIRMACION='AUTORIZO CARGA EN CLON'`. Se crea respaldo previamente. No ejecutar dos instancias simultáneas.

In [ ]:
if not EJECUTAR_ESCRITURA:
 print('SIMULACIÓN: no hubo modificaciones en Google Sheets')
else:
 assert CONFIRMACION=='AUTORIZO CARGA EN CLON','Confirmación requerida'
 assert CLON_ID=='1loeK_IM3RXHaR7CqpjUbz-TsNgOKNrV1ejq8aRC6JyE'
 assert ws.row_values(1)==COLUMNAS
 respaldo=drive.files().copy(fileId=CLON_ID,body={'name':'Respaldo clon '+datetime.datetime.now().isoformat()},fields='id,name,webViewLink').execute()
 assert respaldo.get('id') and respaldo['id']!=CLON_ID
 print('Respaldo:',respaldo.get('webViewLink',respaldo['id']))
 escritos=0
 for i in range(0,len(pendientes),LOTE):
  actual=[codigo(x) for x in ws.col_values(1)[1:] if codigo(x)]
  assert len(actual)==len(set(actual)),'Duplicados detectados, abortado'
  hoy=set(actual);bloque=[r for r in pendientes[i:i+LOTE] if r['codigo'] not in hoy]
  if not bloque:continue
  fila=len(ws.col_values(1))+1
  ws.update(range_name=f'A{fila}:P{fila+len(bloque)-1}',values=[[str(r[c]) for c in COLUMNAS] for r in bloque],value_input_option='RAW')
  verificado={codigo(v[0]) for v in ws.get(f'A{fila}:A{fila+len(bloque)-1}') if v}
  assert verificado=={r['codigo'] for r in bloque},'Verificación del lote falló'
  escritos+=len(bloque);print('Insertados:',escritos)
  time.sleep(1)
 finales=[codigo(v) for v in ws.col_values(1)[1:] if codigo(v)]
 assert len(finales)==len(set(finales))
 assert set(procesos).issubset(set(finales))
 print('CARGA COMPLETADA:',len(finales),'procesos,',escritos,'nuevos')

## Verificación
La segunda carga del mismo archivo debe detectar cero registros nuevos. Revisa `ruta_reporte` y `ruta_csv` en `/content/`. Ofertas, Resultados y OrdenesCompra están fuera del alcance de esta primera versión.